# Task 4 fixed-12-image fitting test
Attach FS2K. No trained checkpoint required. This trains the deeper generator from scratch on 12 fixed training pairs for 1000 total updates. No dropout, augmentation, or discriminator. Debug evidence only. Rerunning resumes its own last.pt automatically; do not substitute other model checkpoints.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
candidates = [p.parent for p in Path('/kaggle/input').rglob('anno_train.json') if (p.parent/'photo').is_dir() and (p.parent/'sketch').is_dir()]
print(candidates)
assert len(candidates)==1, 'Attach one FS2K dataset or set raw to its exact root.'
raw = candidates[0]
subprocess.run([sys.executable,'-m','scripts.prepare_fs2k','--raw-dir',str(raw)],cwd=repo,check=True)


In [ ]:
import shutil
output=Path('/kaggle/working/task4-fixed12')
restore_folder=None # After reset: folder from complete fixed12 backup only.
if restore_folder is not None and not output.exists():
    source=Path(restore_folder)
    assert (source/'last.pt').is_file()
    shutil.copytree(source,output)
subprocess.run([sys.executable,'-u','-m','scripts.overfit_task4_candidate',
 '--raw-dir',str(raw),'--output-dir',str(output),'--steps','1000',
 '--device','cuda','--max-hours','1'],cwd=repo,check=True)


In [ ]:
import json
from IPython.display import display,Image,FileLink
print(json.loads((output/'history.json').read_text())[-1])
display(Image(filename=str(output/'preview.png')))
backup=shutil.make_archive('/kaggle/working/task4-fixed12_backup','zip',root_dir=output.parent,base_dir=output.name)
display(FileLink(backup))
# Keep ZIP locally; share only history.json and preview.png for initial review.
